In [1]:
# importing utils notebook for helper enums and functions
%run utils.ipynb

In [2]:
from math import log2, floor
from pathlib import Path
import random
import networkx as nx

In [3]:
N_INSTANCES = 10

In [4]:
def create_initial_graph(
    graph_model: GraphModel,
    n_nodes: int
) -> nx.Graph:
    if graph_model == GraphModel.gnp:
        return nx.gnp_random_graph(n_nodes, 2*log2(n_nodes) / (n_nodes - 1))
    elif graph_model == GraphModel.ba:
        return nx.barabasi_albert_graph(n_nodes, floor(log2(n_nodes)))
    else:
        raise ValueError(f"Unsupported graph model: {graph_model}")

In [5]:
def create_graph_pair_by_edge_sampling(
    graph: nx.Graph,
    edge_removal_prob: float
) -> Tuple[nx.Graph, nx.Graph]:
    def edge_sampling(graph: nx.Graph, edge_removal_prob: float) -> None:
        for edge in graph.edges:
            if random.uniform(0,1) < edge_removal_prob:
                graph.remove_edge(*edge)

    graph_a: nx.Graph = graph.copy()
    graph_b: nx.Graph = graph.copy()

    edge_sampling(graph=graph_a, edge_removal_prob=edge_removal_prob)
    edge_sampling(graph=graph_b, edge_removal_prob=edge_removal_prob)

    return graph_a, graph_b

In [6]:
def create_instance(
    graph_params: GraphParameters
) -> [nx.Graph, nx.Graph, nx.Graph]:
    original_graph = create_initial_graph(
        graph_model=graph_params.graph_model,
        n_nodes=graph_params.n_nodes
    )
    graph_a, graph_b = create_graph_pair_by_edge_sampling(
        graph=original_graph,
        edge_removal_prob=graph_params.edge_removal_prob
    )
    return original_graph, graph_a, graph_b

In [7]:
def save_graphs(
    graph_input_dir: Path,
    original_graph: nx.Graph,
    graph_a: nx.Graph,
    graph_b: nx.Graph
) -> None:
    graph_input_dir.mkdir(parents=True, exist_ok=True)
    if any(graph_input_dir.iterdir()):
        raise ValueError(f"Cannot save graphs: directory '{graph_input_dir}' is not empty.")
    
    nx.write_edgelist(original_graph, graph_input_dir / GraphFilenames.original, data=False)
    nx.write_edgelist(graph_a, graph_input_dir / GraphFilenames.graph_a, data=False)
    nx.write_edgelist(graph_b, graph_input_dir / GraphFilenames.graph_b, data=False)

In [8]:
# graph props
graph_params = GraphParameters(
    graph_model=GraphModel.gnp,
    n_nodes=2048,
    edge_removal_prob=0.1
)

In [9]:
for instance_idx in range(N_INSTANCES):
    original_graph, graph_a, graph_b = create_instance(
        graph_params=graph_params
    )
    graph_input_dir = get_graph_input_dir(
        graph_params=graph_params,
        instance_idx=instance_idx
    )
    save_graphs(
        graph_input_dir=graph_input_dir,
        original_graph=original_graph,
        graph_a=graph_a,
        graph_b=graph_b
    )